## Using MobileNetV2 for Face Mask Detection

> Importing libraries
---

In [46]:
import pandas as pd
import os
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

> Dataset Inspection
---

In [47]:
df = pd.read_csv("dataset/train/_classes.csv")

print(df.head())
print(df.columns.tolist())
print(df.shape)
print(df.sum(numeric_only=True))

                                            filename   with_mask  \
0  26-with-mask_jpg.rf.fb2a4fd14461ba4e7d981f51d9...           1   
1     66_jpg.rf.fb48333563aba0cd06afba343f625d98.jpg           0   
2  153-with-mask_jpg.rf.fb584ce29ce6d9a3123a04d80...           1   
3  29-with-mask_jpg.rf.fb9ecac8d3dc335a43b79a9438...           1   
4  augmented_image_315_jpg.rf.fba9c25c6675c5a039f...           1   

    without_mask  
0              0  
1              1  
2              0  
3              0  
4              0  
['filename', ' with_mask', ' without_mask']
(1279, 3)
with_mask       644
without_mask    635
dtype: int64


In [48]:
df.columns = df.columns.str.strip()

# Image directory
IMAGE_DIR = "dataset/train/"

# Check image availability
df["exists"] = df["filename"].apply(
    lambda x: os.path.exists(os.path.join(IMAGE_DIR, x))
)

print("Total images:", len(df))
print("Existing images:", df["exists"].sum())
print("Missing images:", (~df["exists"]).sum())

print("\nClass distribution:")
print(df[["with_mask", "without_mask"]].sum())

Total images: 1279
Existing images: 1279
Missing images: 0

Class distribution:
with_mask       644
without_mask    635
dtype: int64


> Train Test Split
---

In [49]:
IMAGE_DIR = "dataset/train"
IMG_SIZE = 160

df = pd.read_csv(os.path.join(IMAGE_DIR, "_classes.csv"))
df.columns = df.columns.str.strip()

X = []
y = []

for _, row in df.iterrows():
    img_path = os.path.join(IMAGE_DIR, row["filename"])

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    X.append(img)

    label = 1 if row["with_mask"] == 1 else 0
    y.append(label)

X = np.array(X, dtype=np.float32) / 255.0
y = np.array(y, dtype=np.int32)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)

Training: (1023, 160, 160, 3) (1023,)
Validation: (256, 160, 160, 3) (256,)


> Data Augmentation
---

In [50]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])

# Base model
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(160, 160, 3)
)

base_model.trainable = False

# Model architecture
inputs = layers.Input(shape=(160, 160, 3))

x = data_augmentation(inputs)
x = layers.Rescaling(scale=2, offset=-1)(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_2 (Sequential)       │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_2 (Rescaling)         │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

> Model Training Optimization & Callbacks
---

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "model/best_mask_model.keras",
        monitor="val_loss",
        save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=callbacks
)

Epoch 1/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 34s 783ms/step - accuracy: 0.8162 - loss: 0.3892 - val_accuracy: 0.9883 - val_loss: 0.1014 - learning_rate: 0.0010
Epoch 2/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 22s 681ms/step - accuracy: 0.9658 - loss: 0.1086 - val_accuracy: 0.9922 - val_loss: 0.0529 - learning_rate: 0.0010
Epoch 3/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 23s 715ms/step - accuracy: 0.9834 - loss: 0.0682 - val_accuracy: 0.9961 - val_loss: 0.0386 - learning_rate: 0.0010
Epoch 4/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 22s 680ms/step - accuracy: 0.9853 - loss: 0.0516 - val_accuracy: 0.9961 - val_loss: 0.0324 - learning_rate: 0.0010
Epoch 5/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 24s 738ms/step - accuracy: 0.9941 - loss: 0.0375 - val_accuracy: 0.9961 - val_loss: 0.0278 - learning_rate: 0.0010
Epoch 6/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 23s 734ms/step - accuracy: 0.9941 - loss: 0.0312 - val_accuracy: 0.9961 - val_loss: 0.0249 - learning_rate: 0.0010
Epoch 7/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 22s 674ms/step - accuracy: 0.9932 - loss: 0.

> Performance Matrix
---

In [52]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = (model.predict(X_val) > 0.5).astype(int)

print(classification_report(
    y_val,
    y_pred,
    target_names=["Without Mask", "With Mask"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))

8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 532ms/step
              precision    recall  f1-score   support

Without Mask       1.00      1.00      1.00       127
   With Mask       1.00      1.00      1.00       129

    accuracy                           1.00       256
   macro avg       1.00      1.00      1.00       256
weighted avg       1.00      1.00      1.00       256

Confusion Matrix:
[[127   0]
 [  0 129]]


> Saving Model
---

In [53]:
model.save("model/mask_detection_model.keras")

>Model Inference Speed Benchmarking
---

In [54]:
import time
import numpy as np

# Warm-up
for _ in range(10):
    model.predict(X_val[:1], verbose=0)

# Benchmark
start = time.perf_counter()

for _ in range(100):
    model.predict(X_val[:1], verbose=0)

end = time.perf_counter()

avg_time = (end - start) / 100

print(f"Average inference time: {avg_time * 1000:.2f} ms")
print(f"Estimated FPS: {1 / avg_time:.2f}")

Average inference time: 102.76 ms
Estimated FPS: 9.73


> Direct Model Inference Benchmarking
---

In [55]:
import time
import tensorflow as tf

sample = tf.convert_to_tensor(X_val[:1])

# Warm-up
for _ in range(20):
    model(sample, training=False)

# Benchmark
start = time.perf_counter()

for _ in range(100):
    model(sample, training=False)

# Ensure execution completes
_ = model(sample, training=False).numpy()

end = time.perf_counter()

avg_time = (end - start) / 100

print(f"Inference: {avg_time * 1000:.2f} ms")
print(f"FPS: {1 / avg_time:.2f}")

Inference: 199.06 ms
FPS: 5.02


> TFLite Model Conversion & Optimization
---

In [56]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("model/mask_detection_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved!")

INFO:tensorflow:Assets written to: C:\Users\myper\AppData\Local\Temp\tmp3ca9m9qx\assets


INFO:tensorflow:Assets written to: C:\Users\myper\AppData\Local\Temp\tmp3ca9m9qx\assets


Saved artifact at 'C:\Users\myper\AppData\Local\Temp\tmp3ca9m9qx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name='keras_tensor_486')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1345217264336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345217265104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345217265296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345217264720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345217263760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345217262992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345217257040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345217263952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345217263184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345217261648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1

>TFLite Inference Performance Benchmarking
---

In [57]:
import numpy as np
import time
import tensorflow as tf

interpreter = tf.lite.Interpreter(
    model_path="model/mask_detection_model.tflite",
    num_threads=4
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample = X_val[:1].astype(np.float32)

interpreter.set_tensor(input_details[0]["index"], sample)

# Warm-up
for _ in range(20):
    interpreter.invoke()

start = time.perf_counter()

for _ in range(100):
    interpreter.set_tensor(input_details[0]["index"], sample)
    interpreter.invoke()

end = time.perf_counter()

avg_time = (end - start) / 100

print(f"Inference: {avg_time * 1000:.2f} ms")
print(f"FPS: {1 / avg_time:.2f}")

d:\CODE 23102026\04-Local Projects\face-mask-detection\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Inference: 24.79 ms
FPS: 40.34


> Keras vs. TFLite Prediction Consistency Check
---

In [58]:
# Keras prediction
keras_pred = model.predict(X_val[:10], verbose=0)

# TFLite prediction
tflite_preds = []

for img in X_val[:10]:
    interpreter.set_tensor(
        input_details[0]["index"],
        np.expand_dims(img, axis=0).astype(np.float32)
    )

    interpreter.invoke()

    pred = interpreter.get_tensor(
        output_details[0]["index"]
    )

    tflite_preds.append(pred[0][0])

print("Keras:", keras_pred.flatten())
print("TFLite:", np.array(tflite_preds))

print(
    "Maximum difference:",
    np.max(np.abs(keras_pred.flatten() - tflite_preds))
)

Keras: [9.7685761e-04 9.9996948e-01 9.9505109e-01 2.0837737e-03 7.0249976e-04
 6.6904793e-04 9.9996650e-01 9.9931329e-01 3.1257799e-04 9.9943036e-01]
TFLite: [8.8427810e-04 9.9995852e-01 9.9436057e-01 1.5735277e-03 6.8302272e-04
 2.8949961e-04 9.9996966e-01 9.9935967e-01 2.3584893e-04 9.9937475e-01]
Maximum difference: 0.0006905198


> Webcame Related Implementations
---

In [59]:

import cv2
import time
import numpy as np
import tensorflow as tf

# Load TFLite model
interpreter = tf.lite.Interpreter(
    model_path="model/mask_detection_model.tflite",
    num_threads=4
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Load face detector
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

# Webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Cannot open webcam")

IMG_SIZE = 160

prev_time = time.perf_counter()

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Improve processing speed
    frame = cv2.flip(frame, 1)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(60, 60)
    )

    for (x, y, w, h) in faces:

        # Extract face
        face = frame[y:y+h, x:x+w]

        # Preprocessing
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))

        face = face.astype(np.float32) / 255.0
        face = np.expand_dims(face, axis=0)

        # TFLite prediction
        interpreter.set_tensor(
            input_details[0]["index"],
            face
        )

        interpreter.invoke()

        prediction = interpreter.get_tensor(
            output_details[0]["index"]
        )[0][0]

        # Classification
        if prediction >= 0.5:
            label = "With Mask"
            color = (0, 255, 0)
            confidence = prediction
        else:
            label = "Without Mask"
            color = (0, 0, 255)
            confidence = 1 - prediction

        # Draw bounding box
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            color,
            2
        )

        cv2.putText(
            frame,
            f"{label}: {confidence*100:.1f}%",
            (x, max(y-10, 25)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            color,
            2
        )

    # FPS calculation
    current_time = time.perf_counter()
    fps = 1 / max(current_time - prev_time, 1e-9)
    prev_time = current_time

    cv2.putText(
        frame,
        f"FPS: {fps:.1f}",
        (15, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 0),
        2
    )

    cv2.imshow("Face Mask Detection", frame)

    # Press Q to exit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


> OpenCV Haar Cascade Face Detector Initialization
---

In [60]:
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

print("Face detector loaded:", not face_detector.empty())

Face detector loaded: True
